# $\beta_{eff}$ - ottobre 2026 (solo $h = 10^{-6}$)

Analisi di $\beta_{eff}$ per il modello di Wilson-Cowan sbilanciato (Nandi et al., *Phys. Rev. E* **107**, 064307 (2023)), con $\chi_E = 70\%$, $\chi_I = 30\%$, **solo per $h = 10^{-6}$**, sulla griglia dei parametri $(\delta_{IE}, \delta_{EI})$.

Le Sez. 1-5 sono la parte su $\beta_{eff}$ del notebook `Wilson_Cowan_ottobre` (ex Sez. 7 e 8), con lo stesso codice; il calcolo di $W$ (Sez. 3) serve solo come maschera. La Sez. 6 (fluttuazione relativa dell'attivita' totale) e' copiata da `Wilson_Cowan_ottobre` (Sez. 9 di quel notebook, Sez. 7 della versione senza $\beta$). La Sez. 7 (relazione tra $\beta_{eff}$ e $W$) viene da `beta_eff_ottobre` (ex Sez. 9 e 9.1) ed e' **da rivedere**.

**Convenzione sui dati.** Tutti i campi vengono caricati dal file `_sigmaFIX.npz` (covarianza $\sigma$ gia' corretta) e arrotondati a `DECIMALI = 9` cifre decimali: valori sotto $10^{-9}$ sono considerati zero. La regione proibita ($w_{EI}<0$ o $w_{IE}<0$) e' messa a NaN.


## 1. Import e parametri

In [12]:
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# FILE E PARAMETRI
# ============================================================
# Solo h = 1e-6 (valore dell'articolo).
files = {
    r"10^{-6}": "grid_dynamics_7030_h1e-06_sigmaFIX.npz",
}

# Parametri fissi del modello (Sez. II e III del paper)
alpha = 0.1
wEE = 6.95
wII = 6.85

chiE = 0.7
chiI = 0.3

# Numero di cifre decimali a cui arrotondare i dati al caricamento
DECIMALI = 9

# ============================================================
# SALVATAGGIO AUTOMATICO DELLE FIGURE
# ============================================================
# Richiede il pacchetto 'kaleido' (pip install kaleido).
CARTELLA_FIGURE = "figure_esportate_ottobre"
os.makedirs(CARTELLA_FIGURE, exist_ok=True)


def salva_figura(fig, nome_file, scale=2):
    """Salva la figura come PNG (CARTELLA_FIGURE/nome_file.png) e poi la mostra."""
    path = os.path.join(CARTELLA_FIGURE, f"{nome_file}.png")
    try:
        fig.write_image(path, scale=scale)
        print(f"Figura salvata: {path}")
    except Exception as e:
        print(f"ATTENZIONE: impossibile salvare '{path}' ({e}). "
              f"Installa il pacchetto con: pip install kaleido")
    fig.show()


def layout_griglia(n_pannelli, max_colonne=2):
    """Numero di righe e colonne per n_pannelli subplot (1 pannello -> 1x1)."""
    n_cols = min(max_colonne, n_pannelli)
    n_rows = int(np.ceil(n_pannelli / n_cols))
    return n_rows, n_cols

## 2. Caricamento dati

In [13]:
# ============================================================
# CARICAMENTO DATI (arrotondati a DECIMALI cifre) + REGIONE PROIBITA
# ============================================================
# delta_IE e delta_EI sono gli assi della griglia: non vengono arrotondati.
# Gli array 2D hanno indice 0 = delta_EI, indice 1 = delta_IE.
#
# Regione proibita (come nelle Fig. 1-2 del paper): w_EI < 0 oppure
# w_IE < 0. 
# Li' tutti i campi 2D vengono messi a NaN.

CAMPI_ASSI = ["delta_IE", "delta_EI"]

data_all = {}

for h_label, filename in files.items():
    raw = np.load(filename)

    entry = {}
    for key in raw.files:
        if key in CAMPI_ASSI:
            entry[key] = raw[key]
        else:
            entry[key] = np.round(raw[key], DECIMALI)

    DEI, DIE = np.meshgrid(entry["delta_EI"], entry["delta_IE"], indexing="ij")
    proibita = ((wII + DEI) < 0) | ((wEE + DIE) < 0)

    for key, val in entry.items():
        if key not in CAMPI_ASSI and np.ndim(val) == 2:
            val = val.astype(float)
            val[proibita] = np.nan
            entry[key] = val

    entry["proibita"] = proibita
    data_all[h_label] = entry

    righe_proibite = entry["delta_EI"][np.any(proibita, axis=1)]
    print(f"h = {h_label}  <-  {filename}")
    print(f"  griglia: {len(entry['delta_EI'])} x {len(entry['delta_IE'])} (delta_EI x delta_IE)")
    print(f"  punti nella regione proibita (messi a NaN): {int(proibita.sum())}, "
          f"righe delta_EI = {np.round(righe_proibite, 2)}")

h = 10^{-6}  <-  grid_dynamics_7030_h1e-06_sigmaFIX.npz
  griglia: 141 x 140 (delta_EI x delta_IE)
  punti nella regione proibita (messi a NaN): 280, righe delta_EI = [-7.  -6.9]


## 3. Tasso di produzione di entropia $W$ (usato come maschera)



$$W = -\frac{\delta^2}{(A_{EE}+A_{II})\,E_0\,I_0}, \qquad \delta = \sqrt{\chi_E/\chi_I}\,A_{EI}\,I_0 - \sqrt{\chi_I/\chi_E}\,A_{IE}\,E_0 .$$

Alla precisione usata ($E_0 < 10^{-9} \to 0$) i punti con $W$ non finito sono di due tipi:

- **$W = 0/0$ (NaN)**: $E_0 = 0$ **e** $A_{EI} = 0$. Primo quadrante: l'input $S_E \le 0$, quindi $f(S_E) = 0$ e $f'(S_E) = 0$; il sistema collassa a 1D (solo neuroni inibitori), e' all'equilibrio e il valore fisico e' $W = 0$. Qui $\beta_{eff}$ non viene calcolato (maschera `W_00`).
- **$W = \delta^2/0 = \infty$**: $E_0 = 0$ ma $A_{EI} \neq 0$. E' **un solo punto** ($\delta_{EI} = 0.1$, $\delta_{IE} = 0$, accanto al punto critico, sul bordo del primo quadrante) ed e' probabilmente un **artefatto numerico**: li' $S_E \approx 6\cdot 10^{-16}$, cioe' zero entro la precisione di macchina, e il segno di $S_E$ decide da solo se $A_{EI} = -w_{EI}$ (punto "2D") o $A_{EI} = 0$ (punto 1D); inoltre $E_0 \approx 3.6\cdot 10^{-12}$ e' rumore del solutore, arrotondato a zero. Va considerato un punto di confine del primo quadrante. Dettagli nel notebook `Wilson_Cowan_ottobre`, Sez. 8. Per ora il punto **non** e' mascherato.

In [14]:
# ============================================================
# TASSO DI PRODUZIONE DI ENTROPIA + CLASSIFICAZIONE DEI PUNTI
# ============================================================

def compute_entropy_rate(data):
    """Ritorna W, numeratore (delta^2), denominatore (Tr(A)*E0*I0) e la maschera 0/0."""
    E0 = data["E0"]; I0 = data["I0"]
    A_EE = data["A_EE"]; A_EI = data["A_EI"]; A_IE = data["A_IE"]; A_II = data["A_II"]

    delta_term = (
        np.sqrt(chiE / chiI) * A_EI * I0
        - np.sqrt(chiI / chiE) * A_IE * E0
    )
    numeratore = delta_term**2
    denominatore = (A_EE + A_II) * E0 * I0

    with np.errstate(divide="ignore", invalid="ignore"):
        W = -numeratore / denominatore

    mask_00 = (numeratore == 0) & (denominatore == 0)
    return W, numeratore, denominatore, mask_00


W_matrices = {}
W_00_masks = {}
W_num = {}
W_den = {}

for h_label, data in data_all.items():
    W, num, den, mask_00 = compute_entropy_rate(data)
    W_matrices[h_label] = W
    W_00_masks[h_label] = mask_00
    W_num[h_label] = num
    W_den[h_label] = den

    valido = ~data["proibita"]
    print(f"h = {h_label}")
    print(f"  W = 0/0 (primo quadrante, 1D, mascherati)   : {int(np.sum(mask_00 & valido))}")
    print(f"  W = inf (delta^2/0, 2D, NON mascherati)     : {int(np.sum(np.isinf(W) & valido))}")
    print(f"  W finito e > 0                               : {int(np.sum(np.isfinite(W) & (W > 0) & valido))}")
    print(f"  W finito e = 0                               : {int(np.sum((W == 0) & valido))}")
    print(f"  regione proibita (NaN)                       : {int(np.sum(data['proibita']))}")

h = 10^{-6}
  W = 0/0 (primo quadrante, 1D, mascherati)   : 6544
  W = inf (delta^2/0, 2D, NON mascherati)     : 1
  W finito e > 0                               : 12915
  W finito e = 0                               : 0
  regione proibita (NaN)                       : 280


---
## 4. $\beta_{eff}$ (temperatura effettiva)

$\beta_{eff} = y(\infty) = \chi_{\Sigma\Sigma}(\infty)/C_{\Sigma\Sigma}(0)$, con $\chi_{\Sigma\Sigma}(\infty) = \int_0^\infty R_{\Sigma\Sigma}(t)\,dt$. Poiche' tutti gli autovalori hanno parte reale negativa, l'integrale ha forma chiusa: $\int_0^\infty e^{At}dt = -A^{-1}$, quindi

$$\beta_{eff} = \frac{-w}{(xw - yz)\,\sigma_{11}} .$$

E' la stessa espressione dei due rami usati prima: nel ramo reale $\frac{\lambda_1(x-\lambda_1)-\lambda_2(x-\lambda_2)}{\lambda_1\lambda_2(\lambda_1-\lambda_2)\sigma_{11}}$, nel ramo complesso $\frac{x-2a}{(a^2+b^2)\sigma_{11}}$; entrambi si riducono a quella sopra (perche' $\lambda_1+\lambda_2 = x+w$, $\lambda_1\lambda_2 = a^2+b^2 = xw-yz$), ma questa non ha il problema di $\sqrt{\Delta}\to 0$ al confine reale/complesso.

**Maschera**: $\beta_{eff}$ viene escluso **solo dove $W = 0/0$** (primo quadrante, sistema 1D). Altrove viene calcolato sempre; dove risulta comunque non finito (es. $\sigma_{11} = 0$ alla precisione usata, oppure $\det A = 0$) semplicemente non viene disegnato, e la cella sotto conta quanti sono questi punti.

Divergenze di $\beta_{eff}$: dove $\det A = xw-yz \to 0$ (punto critico, linea di transizione) e dove $\sigma_{11} \to 0$. Dove $w = 0$, invece, $\beta_{eff} = 0$.

In [15]:
# ============================================================
# beta_eff SU TUTTA LA GRIGLIA, MASCHERATO SOLO DOVE W = 0/0
# ============================================================

def compute_beta_eff_grid(data, mask_W_00):
    x = data["A_x"]; y = data["A_y"]; z = data["A_z"]; w = data["A_w"]
    s11 = data["sigma11"]

    with np.errstate(divide="ignore", invalid="ignore"):
        beta = -w / ((x * w - y * z) * s11)

    beta = np.where(mask_W_00, np.nan, beta)
    return beta


beta_matrices = {}
for h_label, data in data_all.items():
    beta = compute_beta_eff_grid(data, W_00_masks[h_label])
    beta_matrices[h_label] = beta

    valido = ~data["proibita"]
    non_mascherato = valido & ~W_00_masks[h_label]
    non_finito = non_mascherato & ~np.isfinite(beta)
    det = data["A_x"] * data["A_w"] - data["A_y"] * data["A_z"]

    print(f"h = {h_label}")
    print(f"  mascherati (W = 0/0)                     : {int(np.sum(W_00_masks[h_label] & valido))}")
    print(f"  beta_eff finito                          : {int(np.sum(non_mascherato & np.isfinite(beta)))}")
    print(f"  beta_eff non finito (non mascherato)     : {int(non_finito.sum())}")
    print(f"     di cui con sigma11 = 0                : {int(np.sum(non_finito & (data['sigma11'] == 0)))}")
    print(f"     di cui con det(A) = 0                 : {int(np.sum(non_finito & (det == 0)))}")
    print(f"  punti con W = inf e beta_eff finito      : "
          f"{int(np.sum(np.isinf(W_matrices[h_label]) & np.isfinite(beta) & valido))}")

h = 10^{-6}
  mascherati (W = 0/0)                     : 6544
  beta_eff finito                          : 12916
  beta_eff non finito (non mascherato)     : 0
     di cui con sigma11 = 0                : 0
     di cui con det(A) = 0                 : 0
  punti con W = inf e beta_eff finito      : 1


### 4.1 Funzioni di plot

In [16]:
def _beta_scatter_grid(beta_matrices, data_all, title, trace_builder):
    """Scatter interattivi Plotly, un pannello per ciascun h."""
    h_labels = list(beta_matrices.keys())
    n_rows, n_cols = layout_griglia(len(h_labels))

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[f"h={h}" for h in h_labels],
        horizontal_spacing=0.12, vertical_spacing=0.16,
    )
    for idx, h_label in enumerate(h_labels):
        row, col = idx // n_cols + 1, idx % n_cols + 1
        beta = beta_matrices[h_label]
        data = data_all[h_label]
        DEI, DIE = np.meshgrid(data["delta_EI"], data["delta_IE"], indexing="ij")

        traces = trace_builder(h_label, beta, DIE.ravel(), DEI.ravel(), showscale=(idx == 0))
        for tr in traces:
            fig.add_trace(tr, row=row, col=col)

        fig.update_xaxes(title_text="delta_IE", row=row, col=col)
        fig.update_yaxes(title_text="delta_EI", row=row, col=col)

    fig.update_layout(title=title, height=550 * n_rows + 100, width=650 * n_cols + 150)
    return fig


def plot_beta_eff_positive(beta_matrices, data_all, title):
    """beta_eff > 0 colorato, beta_eff <= 0 in nero come sfondo."""

    def builder(h_label, beta, die_flat, dei_flat, showscale):
        beta_flat = beta.ravel()
        mask_pos = np.isfinite(beta_flat) & (beta_flat > 0)
        mask_neg = np.isfinite(beta_flat) & (beta_flat <= 0)

        tr_neg = go.Scattergl(
            x=die_flat[mask_neg], y=dei_flat[mask_neg], mode="markers",
            marker=dict(color="black", size=6), name="beta_eff ≤ 0", showlegend=showscale,
            hovertemplate="delta_IE=%{x:.3g}<br>delta_EI=%{y:.3g}<extra>beta_eff&le;0</extra>",
        )
        tr_pos = go.Scattergl(
            x=die_flat[mask_pos], y=dei_flat[mask_pos], mode="markers",
            marker=dict(color=beta_flat[mask_pos], colorscale="Jet",
                        size=6, showscale=showscale, colorbar=dict(title="beta_eff")),
            hovertemplate=("delta_IE=%{x:.3g}<br>delta_EI=%{y:.3g}<br>"
                           "beta_eff=%{marker.color:.4g}<extra></extra>"),
            showlegend=False,
        )
        return [tr_neg, tr_pos]

    return _beta_scatter_grid(beta_matrices, data_all, title, builder)


def plot_beta_eff_negative(beta_matrices, data_all, title):
    """beta_eff < 0 colorato, beta_eff >= 0 in nero come sfondo."""

    def builder(h_label, beta, die_flat, dei_flat, showscale):
        beta_flat = beta.ravel()
        mask_neg = np.isfinite(beta_flat) & (beta_flat < 0)
        mask_pos = np.isfinite(beta_flat) & (beta_flat >= 0)

        tr_pos = go.Scattergl(
            x=die_flat[mask_pos], y=dei_flat[mask_pos], mode="markers",
            marker=dict(color="black", size=6), name="beta_eff ≥ 0", showlegend=showscale,
            hovertemplate="delta_IE=%{x:.3g}<br>delta_EI=%{y:.3g}<extra>beta_eff&ge;0</extra>",
        )
        tr_neg = go.Scattergl(
            x=die_flat[mask_neg], y=dei_flat[mask_neg], mode="markers",
            marker=dict(color=beta_flat[mask_neg], colorscale="Jet",
                        size=6, showscale=showscale, colorbar=dict(title="beta_eff")),
            hovertemplate=("delta_IE=%{x:.3g}<br>delta_EI=%{y:.3g}<br>"
                           "beta_eff=%{marker.color:.4g}<extra></extra>"),
            showlegend=False,
        )
        return [tr_pos, tr_neg]

    return _beta_scatter_grid(beta_matrices, data_all, title, builder)


def plot_beta_eff_scala_fissa(beta_matrices, data_all, title, vmin=-20, vmax=20):
    """Tutti i punti con beta_eff finito, scala lineare fissa [vmin, vmax].
    I valori fuori scala (vicino a det(A) -> 0 o sigma11 -> 0) appaiono
    saturati al colore limite."""

    def builder(h_label, beta, die_flat, dei_flat, showscale):
        val = beta.ravel()
        mask_valid = np.isfinite(val)
        return [go.Scattergl(
            x=die_flat[mask_valid], y=dei_flat[mask_valid], mode="markers",
            marker=dict(color=val[mask_valid], colorscale="RdBu", cmin=vmin, cmax=vmax,
                        size=6, showscale=showscale, colorbar=dict(title="beta_eff")),
            hovertemplate=("delta_IE=%{x:.3g}<br>delta_EI=%{y:.3g}<br>"
                           "beta_eff=%{marker.color:.4g}<extra></extra>"),
            showlegend=False,
        )]

    return _beta_scatter_grid(beta_matrices, data_all, title, builder)


def plot_beta_eff_log(beta_matrices, data_all, title):
    """log10|beta_eff|, con simbolo diverso per il segno (cerchio > 0, quadrato < 0)."""

    def builder(h_label, beta, die_flat, dei_flat, showscale):
        val = beta.ravel()
        with np.errstate(divide="ignore", invalid="ignore"):
            val_log = np.log10(np.abs(val))
        ok = np.isfinite(val_log)
        simboli = np.where(val[ok] > 0, "circle", "square")
        return [go.Scattergl(
            x=die_flat[ok], y=dei_flat[ok], mode="markers",
            marker=dict(color=val_log[ok], colorscale="Viridis", symbol=simboli,
                        size=6, showscale=showscale, colorbar=dict(title="log10|beta_eff|")),
            customdata=val[ok],
            hovertemplate=("delta_IE=%{x:.3g}<br>delta_EI=%{y:.3g}<br>"
                           "log10|beta_eff|=%{marker.color:.4g}<br>beta_eff=%{customdata:.4g}"
                           "<extra></extra>"),
            showlegend=False,
        )]

    return _beta_scatter_grid(beta_matrices, data_all, title, builder)

### 4.2 Scatter separati: $\beta_{eff}>0$ e $\beta_{eff}<0$ (l'altra parte in nero)

In [17]:
_legend_sotto = dict(orientation="h", yanchor="top", y=-0.08, xanchor="center", x=0.5)

fig = plot_beta_eff_positive(beta_matrices, data_all, "beta_eff > 0 nel piano (delta_IE, delta_EI)")
fig.update_layout(legend=_legend_sotto)
salva_figura(fig, "beta_eff_positivo")

fig = plot_beta_eff_negative(beta_matrices, data_all, "beta_eff < 0 nel piano (delta_IE, delta_EI)")
fig.update_layout(legend=_legend_sotto)
salva_figura(fig, "beta_eff_negativo")

Figura salvata: figure_esportate_ottobre\beta_eff_positivo.png


Figura salvata: figure_esportate_ottobre\beta_eff_negativo.png


### 4.3 $\beta_{eff}$ su scala lineare fissa $[-20,20]$

In [18]:
fig = plot_beta_eff_scala_fissa(beta_matrices, data_all, "beta_eff (scala fissa [-20, 20])", vmin=-20, vmax=20)
salva_figura(fig, "beta_eff_scala_fissa")

Figura salvata: figure_esportate_ottobre\beta_eff_scala_fissa.png


### 4.4 $\beta_{eff}$ in scala logaritmica ($\log_{10}|\beta_{eff}|$, simbolo = segno)

In [19]:
fig = plot_beta_eff_log(beta_matrices, data_all, "log10|beta_eff| (cerchio: > 0, quadrato: < 0)")
salva_figura(fig, "beta_eff_scala_log")

Figura salvata: figure_esportate_ottobre\beta_eff_scala_log.png


## 5. Diagnostica del ramo complesso di $\beta_{eff}$

Scompone $\beta_{eff} = (x-2a)/[\sigma_{11}(a^2+b^2)]$ nei suoi pezzi, solo dove gli autovalori sono complessi. Nota: $x - 2a = -w$ e $a^2+b^2 = \det A$, quindi e' la stessa formula della Sez. 4.

In [20]:
# ============================================================
# DIAGNOSTICA RAMO COMPLESSO: a, b, sigma11, numeratore, denominatore, beta_eff
# ============================================================

h_label = list(data_all.keys())[0]
data = data_all[h_label]

X = data["A_x"]
s11 = data["sigma11"]
a_ = data["lambda_re"]
b_ = data["lambda_im"]
is_cplx = data["is_complex"]

denom = s11 * (a_**2 + b_**2)
numer = X - 2*a_

mask_c = is_cplx == 1

titles = ["a (Re lambda)", "b (Im lambda)", "sigma11", "numeratore (x-2a)", "denominatore", "beta_eff"]
fields = [a_, b_, s11, numer, denom, beta_matrices[h_label]]

fig = make_subplots(rows=2, cols=3, subplot_titles=titles,
                    horizontal_spacing=0.09, vertical_spacing=0.16)

cb_x = [0.30, 0.665, 1.02]
cb_y = [0.80, 0.19]

for k, (title, field) in enumerate(zip(titles, fields)):
    row, col = k // 3 + 1, k % 3 + 1
    f = np.where(mask_c, field, np.nan)
    fig.add_trace(
        go.Heatmap(
            x=data["delta_IE"], y=data["delta_EI"], z=f, colorscale="Viridis",
            hovertemplate=("delta_IE=%{x:.3g}<br>delta_EI=%{y:.3g}<br>"
                           f"{title}=" + "%{z:.6g}<extra></extra>"),
            colorbar=dict(x=cb_x[col - 1], y=cb_y[row - 1], len=0.42, thickness=13),
        ),
        row=row, col=col,
    )
    fig.update_xaxes(title_text="delta_IE", row=row, col=col)
    fig.update_yaxes(title_text="delta_EI", row=row, col=col)

fig.update_layout(title=f"Diagnostica ramo complesso — h = {h_label}", height=800, width=1250)
salva_figura(fig, "diagnostica_ramo_complesso")

Figura salvata: figure_esportate_ottobre\diagnostica_ramo_complesso.png


## 6. Fluttuazione relativa dell'attivita' totale

$\sqrt{\sigma_{11}}/\Sigma_0$. Nota: le fluttuazioni $\xi$ sono gia' scalate per $\sqrt{N}$, quindi la fluttuazione relativa fisica e' $\sqrt{\sigma_{11}/N}/\Sigma_0$. Nel primo quadrante $\sigma_{11} = 0$ alla precisione usata, quindi li' la quantita' non e' definita. Valori molto grandi vicino al punto critico e alla linea di transizione riflettono la divergenza critica ($\det A \to 0$): li' la teoria lineare non e' affidabile.

### 6.0 Funzioni di supporto

Le stesse di `Wilson_Cowan_ottobre` (Sez. 3.0): `plot_quantity` disegna una quantita' di `data_all` nel piano $(\delta_{IE}, \delta_{EI})$ come scatter (default, `TIPO_PLOT_DEFAULT`), heatmap o contour; `log10_positivo` restituisce $\log_{10}$ dove il valore e' positivo e finito, NaN altrove.

In [21]:
# ============================================================
# FUNZIONE DI PLOT GENERICA (scatter / heatmap / contour, INTERATTIVO)
# ============================================================
# Un pannello per ciascun h presente in data_all (qui: uno solo).
# Passando il mouse su un punto compare (delta_IE, delta_EI, valore).

TIPO_PLOT_DEFAULT = "scatter"   # "scatter", "heatmap" oppure "contour"


def plot_quantity(quantity_name, colorbar_label=None, cmap="Jet", levels=50,
                  tipo=None, marker_size=4):
    """
    Plot di una quantita' salvata in data_all, scala colore comune.

    tipo        : "scatter", "heatmap" o "contour" (None -> TIPO_PLOT_DEFAULT)
    levels      : numero di livelli, usato solo da "contour"
    marker_size : dimensione dei marker, usata solo da "scatter"
    """
    tipo = TIPO_PLOT_DEFAULT if tipo is None else tipo
    if tipo not in ("scatter", "heatmap", "contour"):
        raise ValueError("tipo deve essere 'scatter', 'heatmap' o 'contour'")

    all_values = [data[quantity_name] for data in data_all.values()]
    vmin = min(np.nanmin(arr) for arr in all_values)
    vmax = max(np.nanmax(arr) for arr in all_values)

    h_labels = list(data_all.keys())
    n_rows, n_cols = layout_griglia(len(h_labels))

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[f"h={h_label}" for h_label in h_labels],
        horizontal_spacing=0.12, vertical_spacing=0.16,
    )

    label = colorbar_label or quantity_name
    hover = ("delta_IE = %{x:.3g}<br>delta_EI = %{y:.3g}<br>"
             + label + " = %{z:.4g}<extra></extra>")

    for idx, h_label in enumerate(h_labels):
        row, col = idx // n_cols + 1, idx % n_cols + 1
        data = data_all[h_label]
        z = data[quantity_name]

        if tipo == "contour":
            trace = go.Contour(
                x=data["delta_IE"], y=data["delta_EI"], z=z,
                zmin=vmin, zmax=vmax, colorscale=cmap, ncontours=levels,
                showscale=(idx == 0), colorbar=dict(title=label),
                contours=dict(coloring="fill", showlines=False),
                hovertemplate=hover,
            )
        elif tipo == "heatmap":
            trace = go.Heatmap(
                x=data["delta_IE"], y=data["delta_EI"], z=z,
                zmin=vmin, zmax=vmax, colorscale=cmap,
                showscale=(idx == 0), colorbar=dict(title=label),
                hovertemplate=hover,
            )
        else:  # scatter: solo i punti con valore finito
            DEI, DIE = np.meshgrid(data["delta_EI"], data["delta_IE"], indexing="ij")
            z_flat = z.ravel()
            ok = np.isfinite(z_flat)
            trace = go.Scattergl(
                x=DIE.ravel()[ok], y=DEI.ravel()[ok], mode="markers",
                marker=dict(color=z_flat[ok], colorscale=cmap, cmin=vmin, cmax=vmax,
                            symbol="square", size=marker_size,
                            showscale=(idx == 0), colorbar=dict(title=label)),
                hovertemplate=hover.replace("%{z", "%{marker.color"),
                showlegend=False,
            )

        fig.add_trace(trace, row=row, col=col)
        fig.update_xaxes(title_text="delta_IE", row=row, col=col)
        fig.update_yaxes(title_text="delta_EI", row=row, col=col)

    fig.update_layout(title=f"{label}  [{tipo}]",
                      height=450 * n_rows + 100, width=500 * n_cols + 150)
    salva_figura(fig, f"{quantity_name}_{tipo}")


In [22]:
def log10_positivo(val):
    """log10 dove val > 0 (e finito), NaN altrove."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.isfinite(val) & (val > 0), np.log10(np.where(val > 0, val, np.nan)), np.nan)


In [23]:
# ============================================================
# FLUTTUAZIONE RELATIVA: sqrt(sigma11) / Sigma0
# ============================================================

for h_label, data in data_all.items():
    sigma11 = data["sigma11"]
    Sigma0 = data["Sigma0"]

    mask_valid = (
        np.isfinite(sigma11) & (sigma11 >= 0)
        & np.isfinite(Sigma0) & (Sigma0 > 0)
    )

    fluct_rel_Sigma = np.full_like(sigma11, np.nan, dtype=float)
    fluct_rel_Sigma[mask_valid] = np.sqrt(sigma11[mask_valid]) / Sigma0[mask_valid]

    data["fluct_rel_Sigma"] = fluct_rel_Sigma
    data["log10_fluct_rel_Sigma"] = log10_positivo(fluct_rel_Sigma)
    data["log10_sigma11"] = log10_positivo(sigma11)

print("Calcolati: fluct_rel_Sigma, log10_fluct_rel_Sigma, log10_sigma11")

Calcolati: fluct_rel_Sigma, log10_fluct_rel_Sigma, log10_sigma11


In [24]:
plot_quantity("log10_sigma11", colorbar_label="log10(sigma11)")

Figura salvata: figure_esportate_ottobre\log10_sigma11_scatter.png


In [25]:
plot_quantity("log10_fluct_rel_Sigma", colorbar_label="log10[sqrt(sigma11) / Sigma0]")

Figura salvata: figure_esportate_ottobre\log10_fluct_rel_Sigma_scatter.png


In [26]:
# ============================================================
# FLUTTUAZIONE RELATIVA: scatter con scala colore sul vero min/max
# ============================================================

for h_label, d in data_all.items():
    DEI, DIE = np.meshgrid(d["delta_EI"], d["delta_IE"], indexing="ij")
    val_flat = d["log10_fluct_rel_Sigma"].ravel()
    mask = np.isfinite(val_flat)
    vmin, vmax = float(np.min(val_flat[mask])), float(np.max(val_flat[mask]))

    fig = go.Figure(go.Scattergl(
        x=DIE.ravel()[mask], y=DEI.ravel()[mask], mode="markers",
        marker=dict(color=val_flat[mask], colorscale="Jet", cmin=vmin, cmax=vmax,
                    size=5, showscale=True, colorbar=dict(title="log10[sqrt(sigma11)/Sigma0]")),
        hovertemplate=("delta_IE=%{x:.3g}<br>delta_EI=%{y:.3g}<br>"
                       "log10(fluct_rel)=%{marker.color:.3g}<extra></extra>"),
    ))
    fig.update_xaxes(title_text="delta_IE")
    fig.update_yaxes(title_text="delta_EI")
    fig.update_layout(
        title=f"Fluttuazione relativa log10[sqrt(sigma11)/Sigma0] — h={h_label} "
              f"(scala {vmin:.2f} ÷ {vmax:.2f})",
        height=650, width=800,
    )
    salva_figura(fig, "fluttuazione_relativa_scatter")

Figura salvata: figure_esportate_ottobre\fluttuazione_relativa_scatter.png


---
## 7. Relazione tra $\beta_{eff}$ e $W$ (da rivedere)

**Risultato analitico** (sistema lineare a due variabili, rumori di $E$ e $I$ indipendenti con temperature $T_E = \alpha E_0$, $T_I = \alpha I_0$). Nella base $(E,I)$ con i fattori $\sqrt{\chi}$ del paper (Eq. A15),
$$a = A_{EE},\quad b = \sqrt{\chi_E/\chi_I}\,A_{EI},\quad c = \sqrt{\chi_I/\chi_E}\,A_{IE},\quad e = A_{II},$$
si definisce
$$\Lambda = b\,T_I - c\,T_E = \alpha\,\delta ,$$
cioe' (a meno di $\alpha$) lo stesso $\delta$ che compare in $W$. $\Lambda = 0$ e' la condizione di bilancio dettagliato (equilibrio). Allora:
$$W = \frac{\Lambda^2}{T_E\,T_I\,|\mathrm{Tr}A|}, \qquad
\beta_{eff} = \frac{\mathrm{Tr}A\;\bigl(m_2^2\,\Lambda + N_0\bigr)}{2\,m_1 m_2\,\bigl(m_2^2\,\Lambda^2 + B_1\Lambda + B_0\bigr)},$$
con $\Sigma = m_1\xi_E + m_2\xi_I$ (nel paper $m_1 = \chi_E$, $m_2 = \chi_I$; a seconda di come sono normalizzate $\xi_E, \xi_I$ puo' essere $\sqrt{\chi_E}, \sqrt{\chi_I}$: la cella prova entrambe e tiene quella che riproduce i dati) e
$$N_0 = (a+e)\,T_E\,m_1m_2 - b\,(T_E m_1^2 + T_I m_2^2),\qquad B_1 = b\,(T_E m_1^2 - T_I m_2^2) + 2e\,T_E\,m_1m_2,$$
$$B_0 = a^2T_ET_I m_2^2 - 2ab\,T_ET_I m_1m_2 + a\,e\,T_E^2 m_1^2 + a\,e\,T_ET_I m_2^2 - 2b\,e\,T_ET_I m_1m_2 + e^2T_E^2m_1^2 .$$
(I coefficienti $N_0, B_1, B_0$ non contengono $c$: tutta la dipendenza da $c$ e' dentro $\Lambda$.)

**Conseguenze.**
- $W$ dipende da $\Lambda^2$, $\beta_{eff}$ da $\Lambda$ (con il segno): eliminando $\Lambda = \pm\sqrt{W\,T_E T_I |\mathrm{Tr}A|}$ si ottiene $\beta_{eff}$ in funzione di $W$, **ma anche di** $a, b, e, T_E, T_I$ **e del segno di $\delta$**. Non esiste una funzione $\beta_{eff} = F(W)$ valida in tutto il piano: due punti con lo stesso $W$ possono avere $\beta_{eff}$ diversi.
- All'equilibrio ($\Lambda = 0$, $W = 0$) $\beta_{eff}$ non si annulla: vale $\beta_0 = \mathrm{Tr}A\,N_0/(2m_1m_2B_0)$, che dipende dal punto. La quantita' che si annulla insieme a $W$ e' l'**eccesso** $\beta_{eff} - \beta_0$.
- $\beta_{eff}(\Lambda)$ e' un rapporto (numeratore di grado 1, denominatore di grado 2): non e' monotono in $|\Lambda|$ in generale.

La cella: (1) verifica la formula sui dati (confronto con $\beta_{eff}$ della Sez. 4); (2) calcola $\beta_0$ e l'eccesso; (3) correlazione di rango di Spearman tra $W$ e $\beta_{eff}$, $|\beta_{eff}|$, $|\beta_{eff}-\beta_0|$, su tutto il piano e lungo le 12 linee.

In [27]:
# ============================================================
# 7. beta_eff COME FUNZIONE DI Lambda = alpha*delta, E CORRELAZIONE CON W
# ============================================================
import pandas as pd
from IPython.display import display

h_label = r"10^{-6}"
d = data_all[h_label]
W = W_matrices[h_label]
beta = beta_matrices[h_label]

# Matrice di drift in base (E, I) con i fattori sqrt(chi) (Eq. A15) e temperature del rumore
a = d["A_EE"]; e = d["A_II"]
b = np.sqrt(chiE / chiI) * d["A_EI"]
c = np.sqrt(chiI / chiE) * d["A_IE"]
T_E = alpha * d["E0"]; T_I = alpha * d["I0"]
Lam = b * T_I - c * T_E                          # = alpha * delta
TrA = a + e


def beta_da_Lambda(L, m1, m2):
    """beta_eff come funzione esplicita di Lambda (formula del markdown)."""
    N0 = TrA * T_E * m1 * m2 - b * (T_E * m1**2 + T_I * m2**2)
    B1 = b * (T_E * m1**2 - T_I * m2**2) + 2 * e * T_E * m1 * m2
    B0 = (a**2 * T_E * T_I * m2**2 - 2 * a * b * T_E * T_I * m1 * m2 + a * e * T_E**2 * m1**2
          + a * e * T_E * T_I * m2**2 - 2 * b * e * T_E * T_I * m1 * m2 + e**2 * T_E**2 * m1**2)
    with np.errstate(divide="ignore", invalid="ignore"):
        return TrA * (m2**2 * L + N0) / (2 * m1 * m2 * (m2**2 * L**2 + B1 * L + B0))


# (1) Verifica della formula sui dati. Si provano le due convenzioni per Sigma
#     (coefficienti chi o sqrt(chi)) e si tiene quella che riproduce beta_eff.
valido = np.isfinite(beta) & np.isfinite(W)      # nessuna soglia su E0, I0 (solo arrotondamento a 9 cifre)
migliore = None
for nome, (m1, m2) in {"chi": (chiE, chiI), "sqrt(chi)": (np.sqrt(chiE), np.sqrt(chiI))}.items():
    bf = beta_da_Lambda(Lam, m1, m2)
    with np.errstate(divide="ignore", invalid="ignore"):
        diff = np.abs(bf[valido] / beta[valido] - 1)
    med = np.nanmedian(diff)
    print(f"Sigma = m1 xi_E + m2 xi_I con m = {nome:9s}: differenza relativa con beta_eff (Sez. 4): "
          f"mediana {med:.1e}, 95% {np.nanpercentile(diff, 95):.1e}")
    if migliore is None or med < migliore[1]:
        migliore = (nome, med, m1, m2)
nome_conv, med_conv, m1, m2 = migliore
print(f"-> convenzione usata: {nome_conv}"
      + ("   (formula verificata)" if med_conv < 1e-3 else "   ATTENZIONE: la formula non riproduce i dati"))
print("   (nessuna soglia su E0, I0: tutti i punti con W e beta_eff finiti, dati arrotondati a 9 cifre)")

# (2) beta all'equilibrio (Lambda = 0) ed eccesso
beta0 = beta_da_Lambda(np.zeros_like(Lam), m1, m2)
eccesso = beta - beta0

# (3) Correlazione di rango di Spearman con W
df = pd.DataFrame({"W": W[valido], "beta_eff": beta[valido], "|beta_eff|": np.abs(beta[valido]),
                   "|beta_eff - beta0|": np.abs(eccesso[valido])})
df = df[np.isfinite(df).all(axis=1)]
print(f"\nCorrelazione di Spearman su tutto il piano ({len(df)} punti):")
display(pd.DataFrame([{c: df["W"].corr(df[c], method="spearman") for c in ["beta_eff", "|beta_eff|", "|beta_eff - beta0|"]}]).round(3))

valori_linee = [-5, -2.5, -0.5, 0, 2.5, 5]
righe = []
for asse, nome_asse in [("dei", "delta_EI"), ("die", "delta_IE")]:
    for t in valori_linee:
        if asse == "dei":
            i = int(np.argmin(np.abs(d["delta_EI"] - t))); sel = (slice(i, i + 1), slice(None))
        else:
            j = int(np.argmin(np.abs(d["delta_IE"] - t))); sel = (slice(None), slice(j, j + 1))
        m = valido[sel].ravel()
        dl = pd.DataFrame({"W": W[sel].ravel()[m], "b": beta[sel].ravel()[m],
                           "ecc": np.abs(eccesso[sel].ravel()[m])})
        dl = dl[np.isfinite(dl).all(axis=1)]
        righe.append({"linea": f"{nome_asse} = {t}", "n": len(dl),
                      "Sp(W, beta_eff)": dl["W"].corr(dl["b"], method="spearman") if len(dl) > 2 else np.nan,
                      "Sp(W, |beta_eff - beta0|)": dl["W"].corr(dl["ecc"], method="spearman") if len(dl) > 2 else np.nan})
print("Correlazione di Spearman lungo le linee:")
display(pd.DataFrame(righe).round(3))

# Grafico: |beta_eff - beta0| contro W (log-log), colore = segno di delta
ok = np.isfinite(W) & np.isfinite(eccesso) & valido & (W > 0) & (np.abs(eccesso) > 0)
fig = go.Figure(go.Scattergl(
    x=W[ok], y=np.abs(eccesso[ok]), mode="markers",
    marker=dict(size=4, color=np.sign(Lam[ok]), colorscale=[[0, "royalblue"], [1, "crimson"]], opacity=0.6),
    hovertemplate="W=%{x:.3g}<br>|beta_eff - beta0|=%{y:.3g}<extra></extra>"))
fig.update_xaxes(title_text="W", type="log")
fig.update_yaxes(title_text="|β_eff − β_0|", type="log")
fig.update_layout(title="Eccesso di β_eff rispetto all'equilibrio contro W (rosso: δ > 0, blu: δ < 0)",
                  height=550, width=750)
salva_figura(fig, "eccesso_beta_vs_W")

Sigma = m1 xi_E + m2 xi_I con m = chi      : differenza relativa con beta_eff (Sez. 4): mediana 5.1e-09, 95% 1.6e-04
Sigma = m1 xi_E + m2 xi_I con m = sqrt(chi): differenza relativa con beta_eff (Sez. 4): mediana 3.6e-01, 95% 7.2e-01
-> convenzione usata: chi   (formula verificata)
   (nessuna soglia su E0, I0: tutti i punti con W e beta_eff finiti, dati arrotondati a 9 cifre)

Correlazione di Spearman su tutto il piano (12915 punti):


,beta_eff,|beta_eff|,|beta_eff - beta0|
0,-0.893,-0.586,0.818


Correlazione di Spearman lungo le linee:


,linea,n,"Sp(W, beta_eff)","Sp(W, |beta_eff - beta0|)"
0,delta_EI = -5,140,-0.947,0.876
1,delta_EI = -2.5,140,-0.947,0.860
2,delta_EI = -0.5,140,-0.916,0.912
3,delta_EI = 0,140,-0.827,0.957
4,delta_EI = 2.5,51,-1.000,0.888
5,delta_EI = 5,39,-1.000,0.856
6,delta_IE = -5,139,-1.000,1.000
7,delta_IE = -2.5,107,-1.000,0.581
8,delta_IE = -0.5,74,-1.000,0.801
9,delta_IE = 0,69,-1.000,0.853


Figura salvata: figure_esportate_ottobre\eccesso_beta_vs_W.png


### 7.1 Dove vale l'approssimazione lineare (regime $|\beta_{eff}-\beta_0| \propto \sqrt{W}$)

Vicino all'equilibrio $\beta_{eff} \simeq \beta_0 + \beta_1\Lambda$, con
$$\beta_1 = \left.\frac{d\beta_{eff}}{d\Lambda}\right|_{\Lambda=0} = \frac{\mathrm{Tr}A}{2\,m_1 m_2}\;\frac{m_2^2\,B_0 - N_0\,B_1}{B_0^2},$$
e quindi $|\beta_{eff}-\beta_0| \simeq |\beta_1|\sqrt{T_E T_I |\mathrm{Tr}A|}\,\sqrt{W}$.

In ogni punto si confronta l'eccesso esatto $\beta_{eff}-\beta_0$ con quello lineare $\beta_1\Lambda$:
$$\varepsilon = \left|\frac{\beta_{eff}-\beta_0}{\beta_1\Lambda} - 1\right| .$$
Il punto e' nel **regime lineare** se $\varepsilon <$ `TOLL` (5% di default). La cella conta i punti, mostra l'intervallo di $W$ in cui vale, e disegna (1) lo scatter $|\beta_{eff}-\beta_0|$ contro $W$ con i punti colorati secondo il regime, piu' la retta di pendenza $\tfrac12$; (2) la mappa nel piano $(\delta_{IE}, \delta_{EI})$ dei punti nel regime lineare.

In [28]:
# ============================================================
# 7.1 DOVE VALE L'APPROSSIMAZIONE LINEARE beta_eff ~ beta0 + beta1*Lambda
# ============================================================
TOLL = 0.05     # soglia sull'errore relativo dell'eccesso lineare

# Coefficienti della formula (stessi della Sez. 7, con la convenzione scelta m1, m2)
N0 = TrA * T_E * m1 * m2 - b * (T_E * m1**2 + T_I * m2**2)
B1 = b * (T_E * m1**2 - T_I * m2**2) + 2 * e * T_E * m1 * m2
B0 = (a**2 * T_E * T_I * m2**2 - 2 * a * b * T_E * T_I * m1 * m2 + a * e * T_E**2 * m1**2
      + a * e * T_E * T_I * m2**2 - 2 * b * e * T_E * T_I * m1 * m2 + e**2 * T_E**2 * m1**2)
with np.errstate(divide="ignore", invalid="ignore"):
    beta1 = TrA / (2 * m1 * m2) * (m2**2 * B0 - N0 * B1) / B0**2      # d beta_eff / d Lambda in Lambda = 0
    eps = np.abs((beta - beta0) / (beta1 * Lam) - 1)                    # errore relativo dell'eccesso lineare

ok = valido & np.isfinite(eps) & np.isfinite(W) & (W > 0)
lineare = ok & (eps < TOLL)
print(f"Punti considerati: {int(ok.sum())}")
print(f"Regime lineare (errore < {TOLL:.0%}): {int(lineare.sum())} punti ({100 * lineare.sum() / ok.sum():.1f}%)")
if lineare.any():
    print(f"  W nel regime lineare:      da {np.min(W[lineare]):.2e} a {np.max(W[lineare]):.2e}")
    print(f"  W fuori dal regime lineare: da {np.min(W[ok & ~lineare]):.2e} a {np.max(W[ok & ~lineare]):.2e}")
    # coefficiente |beta_eff - beta0| / sqrt(W) nel regime lineare: se e' quasi costante, i punti stanno su una retta
    k = np.abs(beta - beta0)[lineare] / np.sqrt(W[lineare])
    print(f"  |beta_eff - beta0| / sqrt(W) nel regime lineare: mediana {np.median(k):.3g}, "
          f"5-95% [{np.percentile(k, 5):.3g}, {np.percentile(k, 95):.3g}]")

# Coordinate dei punti (per il tooltip)
DEI_g, DIE_g = np.meshgrid(d["delta_EI"], d["delta_IE"], indexing="ij")

# --- (1) scatter |beta_eff - beta0| contro W, colorato per regime ---
ecc = np.abs(beta - beta0)
fig = go.Figure()
for maschera, nome, colore in [(ok & ~lineare, f"non lineare (errore ≥ {TOLL:.0%})", "lightgray"),
                               (lineare, f"regime lineare (errore < {TOLL:.0%})", "seagreen")]:
    m = maschera & (ecc > 0)
    fig.add_trace(go.Scattergl(
        x=W[m], y=ecc[m], mode="markers", name=f"{nome}: {int(m.sum())} punti",
        marker=dict(size=4, color=colore, opacity=0.7),
        customdata=np.stack([eps[m], DIE_g[m], DEI_g[m]], -1),
        hovertemplate=("W=%{x:.3g}<br>|β_eff−β_0|=%{y:.3g}<br>errore=%{customdata[0]:.2%}"
                       "<br>δ_IE=%{customdata[1]:.2f}, δ_EI=%{customdata[2]:.2f}<extra></extra>")))
if lineare.any():
    # retta di pendenza 1/2 con il coefficiente mediano del regime lineare
    xx = np.logspace(np.log10(np.min(W[ok])), np.log10(np.max(W[ok])), 50)
    fig.add_trace(go.Scatter(x=xx, y=np.median(k) * np.sqrt(xx), mode="lines", name="∝ √W (pendenza 1/2)",
                             line=dict(color="black", dash="dash")))
fig.update_xaxes(title_text="W", type="log")
fig.update_yaxes(title_text="|β_eff − β_0|", type="log")
fig.update_layout(title="Eccesso di β_eff contro W: dove vale il regime lineare", height=600, width=850,
                  legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5))
salva_figura(fig, "regime_lineare_scatter")

# --- (2) mappa nel piano: dove vale il regime lineare ---
cat = np.full(W.shape, np.nan)
cat[ok & ~lineare] = 0
cat[lineare] = 1
fig = go.Figure(go.Heatmap(
    x=d["delta_IE"], y=d["delta_EI"], z=cat, zmin=0, zmax=1, hoverongaps=False,
    colorscale=[[0, "lightgray"], [0.5, "lightgray"], [0.5, "seagreen"], [1, "seagreen"]],
    colorbar=dict(tickvals=[0.25, 0.75], ticktext=["non lineare", "lineare"]),
    customdata=eps,
    hovertemplate="δ_IE=%{x:.2f}<br>δ_EI=%{y:.2f}<br>errore=%{customdata:.2%}<extra></extra>"))
fig.update_xaxes(title_text="delta_IE"); fig.update_yaxes(title_text="delta_EI")
fig.update_layout(title=f"Regime lineare |β_eff−β_0| ∝ √W (verde: errore < {TOLL:.0%}; vuoto: escluso)",
                  height=600, width=750)
salva_figura(fig, "regime_lineare_mappa")

Punti considerati: 12915
Regime lineare (errore < 5%): 7615 punti (59.0%)
  W nel regime lineare:      da 3.15e-11 a 1.01e+04
  W fuori dal regime lineare: da 5.50e-02 a 8.29e+04
  |beta_eff - beta0| / sqrt(W) nel regime lineare: mediana 3.61, 5-95% [0.375, 4.11]
Figura salvata: figure_esportate_ottobre\regime_lineare_scatter.png


Figura salvata: figure_esportate_ottobre\regime_lineare_mappa.png
